# Word2Vec Paper Analysis: Efficient Estimation of Word Representations in Vector Space

### Paper Title:Efficient Estimation of Word Representations in Vector Space
Authors: Tomas Mikolov, Kai Chen, Greg Corrado, Jeffrey Dean

## 1. Problem Statement

Most traditional natural language processing (NLP) methods treat each word as a separate, unique symbol—like giving every word its own ID number. This approach has several drawbacks:
- **Sparsity:** The way words are represented (as one-hot vectors) results in huge, mostly empty data structures, making computations inefficient.
- **No semantic relationships:** There’s no built-in way for the model to understand that some words are related in meaning; for example, "king" and "queen" are just as different as "king" and "banana" in these systems.
- **Data inefficiency:** Because the model can’t recognize similarities between words, it needs a lot more data to learn about language patterns.
- **Limited scalability:** Simple methods such as counting word combinations (N-grams), quickly reach their limits, even when using very large amounts of text, and can’t capture deeper relationships or scale well to bigger tasks. While N-gram models can be trained on trillions of words, they fail to capture semantic relationships. For tasks like machine translation or speech recognition where high-quality data is limited (millions to few billions of words), we need more sophisticated representations that can generalize better.

**The Vision:**<br>
The authors aim to create distributed word representations where:
- Similar words have similar vector representations.
- Semantic relationships can be captured through vector arithmetic.
- Models can be trained efficiently on billions of words.
- The representations enable linear algebraic operations like: `vector("King") - vector("Man") + vector("Woman") ≈ vector("Queen")`

## 2. Assumptions, Goals, and Research Questions

### Key Assumptions
1. **Distributional Hypothesis:** Words appearing in similar contexts have similar meanings.
2. **Linear Relationships:** Semantic and syntactic relationships can be captured as linear transformations in vector space.
3. **Scalability Trade-off:** Simpler models trained on more data can outperform complex models on less data.
4. **Context Window Sufficiency:** Local context (few words before/after) contains enough information for learning meaningful representations.

### Research Questions
1. Can we design neural architectures that eliminate computational bottlenecks while preserving representation quality?
2. How do different architectural choices (CBOW vs Skip-gram) affect the types of relationships learned?
3. What's the relationship between training data size, vector dimensionality, and representation quality?
4. Can vector arithmetic capture complex linguistic relationships?

### Goals:
- Propose CBOW and Skip-gram models.
- Train them on very large corpora using efficient optimization.
- Evaluate using a novel test set of word analogies (e.g., Paris:France :: Rome:Italy).

## 3. Mathematical Models and Theoretical Concepts

We want to learn **dense vector representations** (`D`-dimensional embeddings) for each word in a large vocabulary `V`, using an efficient model that can be trained on billions of words.


Training time complexity for any model is given by:
```
O = E × T × Q
```
Where:
- `E` = number of epochs
- `T` = total number of training words (tokens)
- `Q` = computational complexity **per training word**, **depends on the model architecture**

### 3.1 Feedforward NNLM (baseline)

$$
Q = N \times D + N \times D \times H + H \times V
$$

* $N$ = context size (window)
* $D$ = dimension of word vector
* $H$ = hidden units
* $V$ = vocabulary size

**Dominant term:** $H \times V$

### 3.2 Recurrent NNLM (baseline)

$$
Q = H \times H + H \times V
$$

* Avoids fixed context size (RNN uses full history)
* Dominant cost: $H \times H$

### Hierarchical Softmax Optimization
To address the H×V bottleneck, the authors use Huffman tree-based hierarchical softmax:
- Traditional softmax: O(V) operations
- Hierarchical softmax: O(log₂(V)) operations
- Huffman trees assign shorter codes to frequent words
- Achieves ~2× speedup for vocabulary of 1M words

## 4. New Model Architectures


We want to learn a **mathematical model** that represents each word as a **vector in ℝⁿ** such that:

* **Similar words** are **close together** in the vector space.
* These vectors can support operations like:
$$
\text{vec("king")} - \text{vec("man")} + \text{vec("woman")} \approx \text{vec("queen")}
$$

This is known as **distributional semantics**: “You shall know a word by the company it keeps.”

> Now we define some terms:

| Symbol                       | Meaning                                            |
| ---------------------------- | -------------------------------------------------- |
| $V$                          | Vocabulary size (e.g. 10⁵–10⁶)                     |
| $D$                          | Embedding dimension (e.g. 100–300)                 |
| $T$                          | Number of training tokens                          |
| $E$                          | Number of training epochs                          |
| $N$                          | Window size (number of context words on each side) |
| $C$                          | Max skip distance (for Skip-gram)                  |
| $w \in V$                    | A word                                             |
| $\vec{v}_w \in \mathbb{R}^D$ | Embedding of word $w$ (input vector)               |
| $\vec{u}_w \in \mathbb{R}^D$ | Output embedding of word $w$                       |

We want to **learn these vectors** from raw text.


### CBOW MODEL (Continuous Bag of Words)

#### Objective:

Given the **context** words $w_{t-n}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+n}$, **predict** the target (center) word $w_t$.

#### Step-by-step:

#### (a) Input: Context words

Let the context window be size $2n$, centered around word at position $t$. For example:

```text
the cat sat on the mat
         ↑
       center
```

If the center word is "sat", context might be \["the", "cat", "on", "the"].

#### (b) Represent each context word by its vector

Each word $w_i$ in context has an embedding $\vec{v}_{w_i} \in \mathbb{R}^D$.

#### (c) Compute the **mean** context vector:

$$
\vec{h} = \frac{1}{2n} \sum_{i=1}^{2n} \vec{v}_{w_i}
$$

This is a simple **bag-of-words** average — no word order is used.

#### (d) Score for each word $w \in V$:

$$
\text{score}(w) = \vec{u}_w^\top \vec{h}
$$

Here, $\vec{u}_w$ is the **output** embedding of word $w$, and the score is a dot product.

#### (e) Convert scores to probabilities using **softmax**:

$$
P(w_t \mid \text{context}) = \frac{e^{\vec{u}_{w_t}^\top \vec{h}}}{\sum_{w \in V} e^{\vec{u}_w^\top \vec{h}}}
$$

#### (f) Loss function (cross-entropy):

We minimize the **negative log likelihood** of the correct word:

$$
\mathcal{L}_{\text{CBOW}} = -\log P(w_t \mid \text{context})
$$


#### Computational Complexity

Now we will see how to calculate cost per training step:

$$
Q = N \times D + D \times \log_2(V)
$$

* $N \times D$: averaging context vectors
* $D \times \log_2(V)$: computing scores via **hierarchical softmax** (explained later)

So CBOW is **very fast**, especially with hierarchical softmax.


### SKIP-GRAM MODEL

#### Objective:

Given the **center word** $w_t$, predict **each context word** in a window.

> Opposite of CBOW.


### Step-by-step:

#### (a) Input: Center word $w_t$

It has input embedding $\vec{v}_{w_t} \in \mathbb{R}^D$

#### (b) Output: Predict multiple context words $w_{t-n}, \dots, w_{t+n}$

Each has output embedding $\vec{u}_w \in \mathbb{R}^D$

#### (c) For each context word $w_c$, compute:

$$
\text{score}(w_c) = \vec{u}_{w_c}^\top \vec{v}_{w_t}
$$

#### (d) Softmax:

$$
P(w_c \mid w_t) = \frac{e^{\vec{u}_{w_c}^\top \vec{v}_{w_t}}}{\sum_{w \in V} e^{\vec{u}_w^\top \vec{v}_{w_t}}}
$$

#### (e) Loss:

We sum over all context words:

$$
\mathcal{L}_{\text{SG}} = -\sum_{c \in \text{context}} \log P(w_c \mid w_t)
$$



#### Computational Complexity

$$
Q = C \times (D + D \cdot \log_2(V))
$$

* $C$: number of context words
* $D$: vector dimension
* $\log_2(V)$: via hierarchical softmax

#### FINAL OBJECTIVE FOR TRAINING

For both models, we use:
* **Stochastic Gradient Descent (SGD)**
* **Backpropagation** to update both $\vec{v}_w$ and $\vec{u}_w$
* **Loss = sum over all training positions** of the negative log-likelihood



## 5. Methodology Deep Dive

### Training Process
1. **Data preprocessing**: Restrict vocabulary to most frequent words
2. **Context extraction**: Sliding window over text corpus
3. **Stochastic gradient descent**: With linear learning rate decay
4. **Hierarchical softmax**: For efficient probability computation

### Architecture Comparison Strategy
The authors use a systematic approach:
1. **Controlled experiments**: Same data, same dimensionality across models
2. **Computational complexity analysis**: Focus on operations that scale with vocabulary size
3. **Quality metrics**: Comprehensive evaluation on syntactic/semantic tasks

### Distributed Training (DistBelief)
- **Asynchronous SGD**: Multiple model replicas with gradient synchronization
- **Adaptive learning rates**: Using Adagrad optimization
- **Massive scale**: 100+ replicas across data center machines


## 6. COMPARISON WITH BASELINE METHODS

| Model         | Semantic Accuracy | Syntactic Accuracy | Total   |
| ------------- | ----------------- | ------------------ | ------- |
| RNNLM         | 9%                | 36%                | 35%     |
| NNLM          | 23%               | 53%                | 47%     |
| **CBOW**      | 24%               | **64%**            | 61%     |
| **Skip-gram** | **55%**           | 59%                | **56%** |


> We can see that skip-gram dominates in **semantic** tasks, CBOW is slightly better on **syntactic**.



## 7. Experimental Setup and Evaluation

### Datasets
- **Google News corpus**: 6 billion tokens for large-scale experiments
- **LDC corpora**: 320M words for controlled comparisons
- **Vocabulary**: Restricted to 1M most frequent words

### Evaluation Framework

#### Semantic-Syntactic Word Relationship Test
- **Semantic questions**: 8,869 questions (5 types)
  - Capital cities: Athens:Greece :: Oslo:Norway
  - Currency: Angola:kwanza :: Iran:rial
  - City-state: Chicago:Illinois :: Stockton:California
  - Gender: brother:sister :: grandson:granddaughter
  
- **Syntactic questions**: 10,675 questions (9 types)
  - Comparative: great:greater :: tough:tougher
  - Superlative: easy:easiest :: lucky:luckiest
  - Present participle: think:thinking :: read:reading
  - Past tense: walking:walked :: swimming:swam
  - Plural: mouse:mice :: dollar:dollars

#### Evaluation Method
- **Vector arithmetic**: `vec(b) - vec(a) + vec(c) ≈ vec(d)`
- **Nearest neighbor search**: Find closest word to computed vector
- **Strict accuracy**: Only exact matches count (synonyms are errors)
- **Cosine similarity**: Distance metric for vector comparisons

### Additional Benchmarks
- **Microsoft Sentence Completion Challenge**: 1,040 sentences with missing words
- **MSR Word Relatedness Test**: Syntactic similarity benchmark

